In [13]:
import os
import tempfile
from data_manager import DataManager
from get_props import process_all_csv_files_in_directory

In [14]:
def open_df_in_temp_excel(df):
    # Create a temporary file with .xlsx extension
    with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as temp_file:
        temp_file_path = temp_file.name
    
    # Save the DataFrame to the temporary Excel file
    df.to_excel(temp_file_path, index=False, engine='openpyxl')
    
    # Open the Excel file
    os.startfile(temp_file_path)

    # Print path for reference
    print(f"Temporary Excel file opened at: {temp_file_path}")

In [34]:
def calculate_percentiles(df, stat_column, percentiles=[0.25, 0.5, 0.75]):
    """
    Calculate specified percentiles for a given stat column in a DataFrame.

    Parameters:
    df (pd.DataFrame): DataFrame containing the player's game log.
    stat_column (str): The column name for the stat to calculate percentiles on.
    percentiles (list): A list of percentiles to calculate (default is [0.25, 0.5, 0.75]).

    Returns:
    pd.Series: Percentile values for the specified stat column.
    """
    if stat_column not in df.columns:
        raise ValueError(f"Column '{stat_column}' not found in DataFrame.")
    
    # Calculate the percentiles
    percentile_values = df[stat_column].quantile(percentiles)
    
    return percentile_values


def calculate_multiple_emas(df, stat_column, spans=[5, 10, 20]):
    """
    Calculate three EMAs for a given stat column in a DataFrame with specified spans.

    Parameters:
    df (pd.DataFrame): DataFrame containing the player's game log.
    stat_column (str): The column name for the stat to calculate EMAs on.
    spans (list): A list of three spans for calculating EMAs (default is [5, 10, 20]).

    Returns:
    pd.DataFrame: DataFrame with additional columns for each EMA.
    """
    df = df.copy()[::-1]
    if len(spans) != 3:
        raise ValueError("Please provide exactly three span values.")
    if stat_column not in df.columns:
        raise ValueError(f"Column '{stat_column}' not found in DataFrame.")
    
    # Calculate each EMA and add it to the DataFrame
    for _, span in enumerate(spans):
        ema_column = f"{stat_column}_ema_{span}"
        df[ema_column] = df[stat_column].ewm(span=span, adjust=False).mean()
    df = df.copy()[::-1]
    
    return df

In [3]:
dm = DataManager()

In [4]:
data_path = r"E:\coding_projects\nba_01\prop_lines\11032024"

In [5]:
data = process_all_csv_files_in_directory(data_path)

E:\coding_projects\nba_01\prop_lines\11032024\ATL NO.csv
E:\coding_projects\nba_01\prop_lines\11032024\DET IND.csv
E:\coding_projects\nba_01\prop_lines\11032024\ORL DAL.csv


In [7]:
display(data)

,player_name,team,stat,over_threshold,over_odds,under_threshold,under_odds
0,Clint Capela,Hawks,points,10.5,-120,10.5,-110
1,Jalen Johnson,Hawks,points,19.5,-115,19.5,-115
2,Trae Young,Hawks,points,27.5,-115,27.5,-115
3,Zaccharie Risacher,Hawks,points,11.5,-105,11.5,-125
4,Jose Alvarado,Pelicans,points,10.5,-130,10.5,-105
...,...,...,...,...,...,...,...
77,Jalen Suggs,Magic,rebounds,4.5,-150,4.5,105
78,Anthony Black,Magic,rebounds,2.5,-165,2.5,115
79,Kyrie Irving,Mavericks,rebounds,4.5,100,4.5,-145
80,Daniel Gafford,Mavericks,rebounds,6.5,-105,6.5,-140


In [8]:
players = data['player_name'].unique()
game_data_players = []
for player in players:
    player_id = dm.get_player_id(player)
    player_game_data = dm.get_and_save_player_data(player_id)
    game_data_players.append(player_game_data)

teams = data['team'].unique()
game_data_teams = []
for team in teams:
    team_id = dm.get_team_id(team)
    team_game_data = dm.get_and_save_team_data(team_id)
    game_data_teams.append(team_game_data)

game_data = dict(zip(players, game_data_players))
for i, team in enumerate(teams):
    game_data[team] = game_data_teams[i]

In [42]:
print(teams)

['Hawks' 'Pelicans' 'Pistons' 'Nets' 'Magic' 'Mavericks']


In [43]:
selected_player = "Hawks"
stat_of_interest = "points"
df = game_data[selected_player]
percentiles = tuple(calculate_percentiles(df, stat_of_interest))
emas = calculate_multiple_emas(df, stat_of_interest)

In [44]:
open_df_in_temp_excel(emas)

Temporary Excel file opened at: C:\Users\rusta\AppData\Local\Temp\tmpsabbehwt.xlsx


In [23]:
display(res)

0.25    19.0
0.50    25.0
0.75    31.0
Name: points, dtype: float64